In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

*ประมาณการใช้งาน: หนึ่งนาทีบนโปรเซสเซอร์ Eagle (หมายเหตุ: นี่เป็นเพียงการประมาณการเท่านั้น เวลาจริงอาจแตกต่างออกไป)*

## พื้นหลัง

Circuit-knitting เป็นคำรวมที่ครอบคลุมวิธีการต่างๆ ในการแบ่ง Circuit ออกเป็น subcircuit ขนาดเล็กหลายๆ ชิ้น ซึ่งประกอบด้วย Gate และ/หรือ Qubit น้อยลง แต่ละ subcircuit สามารถรันได้อิสระ และผลลัพธ์สุดท้ายได้จากการประมวลผลแบบคลาสสิกผ่านผลลัพธ์ของแต่ละ subcircuit เทคนิคนี้เข้าถึงได้ผ่าน [Qiskit addon สำหรับการตัด Circuit](https://qiskit.github.io/qiskit-addon-cutting/index.html) โดยมีคำอธิบายอย่างละเอียดของเทคนิคนี้อยู่ใน [docs](https://qiskit.github.io/qiskit-addon-cutting/explanation/index.html) พร้อมด้วย [เนื้อหาเบื้องต้นอื่นๆ](https://qiskit.github.io/qiskit-addon-cutting/tutorials/index.html)

โน้ตบุ๊กนี้เกี่ยวกับวิธีที่เรียกว่า <b>wire cutting</b> ซึ่ง Circuit ถูกแบ่งตามสายสัญญาณ [\[1\], \[2\]](#references) สังเกตว่าในวงจรคลาสสิก การแบ่งนั้นง่ายเพราะผลลัพธ์ที่จุดแบ่งสามารถกำหนดได้แน่นอน คือ 0 หรือ 1 อย่างไรก็ตาม สถานะของ Qubit ที่จุดตัดนั้น โดยทั่วไปจะเป็น mixed state ดังนั้นแต่ละ subcircuit จึงต้องวัดหลายครั้งในฐาน (basis) ที่แตกต่างกัน (โดยทั่วไปจะเป็นชุด basis ที่ครบถ้วนสำหรับ tomography เช่น Pauli basis [\[3\], \[4\]](#references)) และเตรียมใน eigenstate ที่สอดคล้องกัน ภาพด้านล่าง (<i>ที่มา: PhD Thesis, Ritajit Majumdar</i>) แสดงตัวอย่างของ wire cutting สำหรับ GHZ state ขนาด 4 Qubit ออกเป็นสาม subcircuit ที่นี่ $M_j$ แทนชุดของ basis (โดยทั่วไปคือ Pauli X, Y และ Z) และ $P_i$ แทนชุดของ eigenstate (โดยทั่วไปคือ $|0\rangle$, $|1\rangle$, $|+\rangle$ และ $|+i\rangle$)

![wc-1.png](../docs/images/tutorials/wire-cutting-to-improve-performance/0ce8857b-7f5f-400e-8536-6a496c724d50.avif)
![wc-2.png](../docs/images/tutorials/wire-cutting-to-improve-performance/cbce4455-4794-4c81-8630-3e3993e1b29f.avif)

เนื่องจากแต่ละ subcircuit มี Qubit และ/หรือ Gate น้อยกว่า จึงคาดว่าจะได้รับผลกระทบจากสัญญาณรบกวนน้อยลง โน้ตบุ๊กนี้แสดงตัวอย่างที่วิธีนี้สามารถใช้เพื่อลดสัญญาณรบกวนในระบบได้อย่างมีประสิทธิภาพ

## สิ่งที่ต้องการ

ก่อนเริ่มบทเรียนนี้ ต้องติดตั้งสิ่งต่อไปนี้:

- Qiskit SDK v2.0 หรือใหม่กว่า พร้อมรองรับ [visualization](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 หรือใหม่กว่า ( `pip install qiskit-ibm-runtime` )
- Circuit cutting Qiskit addon v0.9.0 หรือใหม่กว่า (`pip install qiskit-addon-cutting`)

เราจะพิจารณาวงจร Many Body Localization (MBL) สำหรับโน้ตบุ๊กนี้ วงจร MBL เป็นวงจรที่มีประสิทธิภาพสำหรับฮาร์ดแวร์และมีพารามิเตอร์สองตัวคือ $\theta$ และ $\vec{\phi}$ เมื่อกำหนด $\theta$ เป็น $0$ และเตรียมสถานะเริ่มต้นใน $|0\rangle$ สำหรับ Qubit ทั้งหมด ค่าความคาดหวังในอุดมคติของ $\langle Z_i \rangle$ จะเท่ากับ $+1$ สำหรับทุก Qubit ตำแหน่ง $i$ โดยไม่ขึ้นกับค่าของ $\vec{\phi}$ สามารถดูรายละเอียดเพิ่มเติมเกี่ยวกับวงจร MBL ได้ใน <a href="https://arxiv.org/abs/2307.07552">บทความนี้</a>

## ตั้งค่า

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.quantum_info import PauliList, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit.result import sampled_expectation_value

from qiskit_addon_cutting.instructions import CutWire
from qiskit_addon_cutting import (
    cut_wires,
    expand_observables,
    partition_problem,
    generate_cutting_experiments,
    reconstruct_expectation_values,
)

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2, Batch

## ส่วนที่ I. ตัวอย่างขนาดเล็ก

### ขั้นตอนที่ 1: แมปอินพุตแบบคลาสสิกไปยังปัญหาควอนตัม

ก่อนอื่นเราสร้าง Circuit แม่แบบโดยไม่มีค่าพารามิเตอร์ที่เฉพาะเจาะจง เรายังเพิ่ม placeholder ที่เรียกว่า `CutWire` เพื่อระบุตำแหน่งของการตัด สำหรับตัวอย่างขนาดเล็ก เราพิจารณาวงจร MBL ขนาด 10 Qubit

In [2]:
class MBLChainCircuit(QuantumCircuit):
    def __init__(
        self, num_qubits: int, depth: int, use_cut: bool = False
    ) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainCircuit<{num_qubits}, {depth}>"
        )
        evolution = MBLChainEvolution(num_qubits, depth, use_cut)
        self.compose(evolution, inplace=True)


class MBLChainEvolution(QuantumCircuit):
    def __init__(self, num_qubits: int, depth: int, use_cut) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainEvolution<{num_qubits}, {depth}>"
        )

        theta = Parameter("θ")
        phis = ParameterVector("φ", num_qubits)

        for layer in range(depth):
            layer_parity = layer % 2
            # print("layer parity", layer_parity)
            for qubit in range(layer_parity, num_qubits - 1, 2):
                # print(qubit)
                self.cz(qubit, qubit + 1)
                self.u(theta, 0, np.pi, qubit)
                self.u(theta, 0, np.pi, qubit + 1)
                if (
                    use_cut
                    and layer_parity == 0
                    and (
                        qubit == num_qubits // 2 - 1
                        or qubit == num_qubits // 2
                    )
                ):
                    self.append(CutWire(), [num_qubits // 2])
                if use_cut and layer < depth - 1 and layer_parity == 1:
                    if qubit == num_qubits // 2:
                        self.append(CutWire(), [qubit])
            for qubit in range(num_qubits):
                self.p(phis[qubit], qubit)

In [3]:
num_qubits = 10
depth = 2
mbl = MBLChainCircuit(num_qubits, depth)
mbl.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

We calculate the average expectation value $O = \frac{1}{n} \sum_i Z_i$ over all qubits for $\theta = 0$. Since the ideal expectation value of $\langle Z_i \rangle = 1$ $\forall$ $i$, the ideal expectation value of $O$ is also $1$. The parameters $\phi$ are selected randomly.

In [4]:
np.random.seed(42)
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

ตอนนี้เราระบุตำแหน่งการตัดใน Circuit โดยแทรก **CutWire** ที่เหมาะสมเพื่อสร้างการตัดที่มีขนาดใกล้เคียงกันสองส่วน เรากำหนด `use_cut=True` ในฟังก์ชัน และให้มันระบุตำแหน่งหลังจาก $\frac{n}{2}$ Qubit โดยที่ $n$ คือจำนวน Qubit ในวงจรดั้งเดิม

In [5]:
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_cut.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/31844134-514b-46ea-85f9-133e432f053f-0.avif)

### ขั้นตอนที่ 2: ปรับ Circuit ให้เหมาะสมสำหรับการรันบนฮาร์ดแวร์ควอนตัม
ต่อไปเราตัด Circuit ออกเป็น subcircuit ขนาดเล็ก สองชิ้น สำหรับตัวอย่างนี้เราจำกัดที่ 2 subcircuit เท่านั้น สำหรับสิ่งนี้ เราใช้ <a href="https://qiskit.github.io/qiskit-addon-cutting/">Qiskit Addon: Circuit Cutting</a>
#### ตัด Circuit ออกเป็น subcircuit ขนาดเล็ก
การตัดสายสัญญาณที่จุดหนึ่งจะเพิ่มจำนวน Qubit ขึ้นหนึ่งตัว นอกจาก Qubit เดิมแล้ว ยังมี Qubit เพิ่มเติมอีกหนึ่งตัวเป็น placeholder สำหรับ Circuit หลังการตัด ภาพต่อไปนี้แสดงให้เห็นถึงสิ่งนี้:

![wc-4.png](../docs/images/tutorials/wire-cutting-to-improve-performance/dfc5f923-e507-4873-888e-d90e1618be3a.avif)

Addon นี้ใช้ฟังก์ชัน `cut_wires` เพื่อรองรับ Qubit เพิ่มเติมที่เกิดจากการตัด

In [6]:
mbl_move = cut_wires(mbl_cut)
mbl_move.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1834cb22-0.avif" alt="Output of the previous code cell" />

#### สร้างและขยาย observable
ตอนนี้เราสร้าง observable $M_z = \frac{1}{n}\sum_{i=1}^n \langle Z_i \rangle$ เนื่องจากผลลัพธ์ในอุดมคติของ $\langle Z_i \rangle$ สำหรับแต่ละ $i$ คือ $+1$ ดังนั้นผลลัพธ์ในอุดมคติของ $M_z$ ก็คือ $+1$ เช่นกัน

In [7]:
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
observable

PauliList(['ZIIIIIIIII', 'IZIIIIIIII', 'IIZIIIIIII', 'IIIZIIIIII',
           'IIIIZIIIII', 'IIIIIZIIII', 'IIIIIIZIII', 'IIIIIIIZII',
           'IIIIIIIIZI', 'IIIIIIIIIZ'])

In [8]:
new_obs = expand_observables(observable, mbl, mbl_move)
new_obs

PauliList(['ZIIIIIIIIII', 'IZIIIIIIIII', 'IIZIIIIIIII', 'IIIZIIIIIII',
           'IIIIZIIIIII', 'IIIIIIZIIII', 'IIIIIIIZIII', 'IIIIIIIIZII',
           'IIIIIIIIIZI', 'IIIIIIIIIIZ'])

อย่างไรก็ตาม สังเกตว่าจำนวน Qubit ใน Circuit ได้เพิ่มขึ้นหลังจากแทรกการดำเนินการ `Move` เสมือน 2 Qubit หลังการตัด ดังนั้นเราจึงต้องขยาย observable ด้วยโดยแทรก identity เพื่อให้ตรงกับ Circuit ปัจจุบัน

In [12]:
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

Here we visualize the two subcircuits:

In [13]:
subcircuits[0].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1b0e779f-0.avif" alt="Output of the previous code cell" />

In [14]:
subcircuits[1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/3c802f28-0.avif" alt="Output of the previous code cell" />

มาดู subcircuit กัน

In [15]:
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

As discussed before, for each cut the upstream circuit must be measured in a Pauli basis, and the downstream circuit must be prepared in the eigenstate of the basis. The function `generate_cutting_experiments` creates all of these necessary circuits and the coefficients associated with each circuit required for reconstruction. Find more detail in [this paper](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.125.150504).

In [16]:
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

#### Transpile the circuits onto the backend

For the first example involving only simulation, we transpile the circuit into the basis gate set of the backend:

In [17]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)

print(backend)

<IBMBackend('ibm_fez')>


![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/35920640-76e8-4af6-a252-ee6a22e9c26a-0.avif)

observable ก็ถูกแบ่งด้วยเพื่อให้เข้ากับ subcircuit

In [20]:
pm_basis = generate_preset_pass_manager(
    optimization_level=2, basis_gates=backend.configuration().basis_gates
)
basis_subexperiments = {
    label: pm_basis.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

In [21]:
sampler = SamplerV2(mode=AerSimulator())
jobs = {
    label: sampler.run(subsystem_subexpts, shots=2**12)
    for label, subsystem_subexpts in basis_subexperiments.items()
}

สังเกตว่าแต่ละ subcircuit จะสร้างตัวอย่างจำนวนหนึ่ง การ reconstruct จะนำผลลัพธ์ของแต่ละตัวอย่างเหล่านี้มาคำนึงถึง แต่ละตัวอย่างเหล่านี้เรียกว่า `subexperiment`
การขยาย observable โดยใช้การดำเนินการ `Move` ต้องใช้โครงสร้างข้อมูล `PauliList` เราสามารถสร้าง observable $M_z$ ในโครงสร้างข้อมูล `SparsePauliOp` ที่มีความยืดหยุ่นมากกว่า ซึ่งจะมีประโยชน์ในภายหลังระหว่างการ reconstruct subexperiment

In [22]:
# Retrieve results
results = {label: job.result() for label, job in jobs.items()}

In [23]:
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real
reconstructed_expval

np.float64(0.9953821063041687)

In [32]:
methods = [
    "Uncut",
    "Wire cut",
]
values = [
    1,
    reconstructed_expval,
]  # since the ideal expectation value in noiseless simulation is +1

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

มาดูตัวอย่างสองตัวอย่างที่ Qubit ที่ถูกตัดถูกวัดใน basis สองแบบที่แตกต่างกัน ครั้งแรกวัดใน basis Z ปกติ และครั้งต่อไปวัดใน basis X

In [ ]:
num_qubits = 60
depth = 2

# construct the circuit
mbl = MBLChainCircuit(num_qubits, depth)

# create parameters
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

# construct the cut circuit
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_move = cut_wires(mbl_cut)

# Define observable and expand to account for the wire cut
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
new_obs = expand_observables(observable, mbl, mbl_move)

# Construct a SparsePauliOp version of the observable for later use in reconstruction
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

# Partition the circuit and get subcircuits and subobservables
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

# Obtain subexperiments and coefficients
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

# Transpile the subexperiments to the backend
pm = generate_preset_pass_manager(optimization_level=2, backend=backend)
isa_subexperiments = {
    label: pm.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

# Execute the subexperiments and retrieve results
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    sampler.options.environment.job_tags = ["TUT_WC"]
    jobs = {
        label: sampler.run(subsystem_subexpts, shots=2**12)
        for label, subsystem_subexpts in isa_subexperiments.items()
    }
results = {label: job.result() for label, job in jobs.items()}

# Reconstruct the expectation value of the original observable
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real

# Compute the uncut circuit to obtain the noisy expectation value for comparison
sampler = SamplerV2(mode=backend)
sampler.options.environment.job_tags = ["TUT_WC"]

if mbl.num_clbits == 0:
    mbl.measure_all()
isa_mbl = pm.run(mbl)

pub = (isa_mbl, params)
uncut_job = sampler.run([pub])

uncut_counts = uncut_job.result()[0].data.meas.get_counts()
uncut_expval = sampled_expectation_value(uncut_counts, M_z)

# visualize the results
ax = plt.gca()
methods = ["uncut", "cut"]
values = [uncut_expval, reconstructed_expval]

plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.axhline(y=1, color="k", linestyle="--")
plt.text(0.3, 0.95, "Exact result")
plt.show()

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/37834c72-0.avif" alt="Output of the previous code cell" />

In [29]:
uncut_expval

0.9202473958333336

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/987547e4-296a-41e4-ad82-41f4139a87a0-0.avif)

#### Transpile แต่ละ subexperiment

ตอนนี้เราต้องทำการ Transpile Circuit ก่อนส่งไปรันจริง ดังนั้นเราจะ Transpile แต่ละ Circuit ใน subexperiment ก่อน